In [2]:
!pip install yfinance arch python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 20.9 MB/s eta 0:00:00


In [3]:
import yfinance as yf
import numpy as np
import pandas as pd
from arch import arch_model
from docx import Document

# 1. Download MMM Data (Jan 1, 2000 to Dec 31, 2010)
mmm = yf.download("MMM", start="2000-01-01", end="2010-12-31")

# Flatten columns in case yfinance returns a MultiIndex
if isinstance(mmm.columns, pd.MultiIndex):
    mmm.columns = mmm.columns.get_level_values(0)

# 2. Calculate Daily Percentage Returns (Log-Difference)
mmm['Log_Return'] = 100 * (np.log(mmm['Close']) - np.log(mmm['Close'].shift(1)))
returns_mmm = mmm['Log_Return'].dropna()

# 3. Estimate AR(0)-ARCH(10) and AR(0)-ARCH(20) Models
arch10_res = arch_model(returns_mmm, mean='Constant', vol='ARCH', p=10).fit(disp='off')
arch20_res = arch_model(returns_mmm, mean='Constant', vol='ARCH', p=20).fit(disp='off')

# Extract AIC and BIC
aic_10, bic_10 = arch10_res.aic, arch10_res.bic
aic_20, bic_20 = arch20_res.aic, arch20_res.bic

# Programmatically determine the preferred models (lower is better)
preferred_aic = "ARCH(10)" if aic_10 < aic_20 else "ARCH(20)"
preferred_bic = "ARCH(10)" if bic_10 < bic_20 else "ARCH(20)"

# ---------------------------------------------------------
# 4. Export Results and Interpretations to Word
# ---------------------------------------------------------
doc = Document()
doc.add_heading('3M Co. (MMM) ARCH Model Selection (2000-2010)', 0)

# Section 1: The Raw Scores
doc.add_heading('1. Model Estimation Results', level=1)
doc.add_paragraph(
    f"AR(0)-ARCH(10) Model:\n"
    f"• AIC: {aic_10:.2f}\n"
    f"• BIC: {bic_10:.2f}"
)
doc.add_paragraph(
    f"AR(0)-ARCH(20) Model:\n"
    f"• AIC: {aic_20:.2f}\n"
    f"• BIC: {bic_20:.2f}"
)

# Section 2: The Interpretation
doc.add_heading('2. Model Selection Criteria Interpretation', level=1)
doc.add_paragraph(
    f"AIC Preference: The model preferred by the Akaike Information Criterion (AIC) is the {preferred_aic}, "
    f"because it has the lower AIC value. The AIC evaluates the goodness of fit while applying a moderate "
    f"penalty for adding more parameters. In financial time series, it often favors models with more lags "
    f"if they capture additional volatility dynamics."
)
doc.add_paragraph(
    f"BIC Preference: The model preferred by the Bayesian Information Criterion (BIC) is the {preferred_bic}, "
    f"because it has the lower BIC value. The BIC applies a much stricter mathematical penalty for complexity "
    f"than the AIC. Because of this harsh penalty for adding 10 extra lags, the BIC heavily favors the more "
    f"parsimonious (simpler) model."
)

# Save the document
filename = 'MMM_ARCH_Model_Selection.docx'
doc.save(filename)
print(f"Success! Results and interpretations saved to {filename}")

# If you are using Google Colab, this will automatically download the file to your computer!
try:
    from google.colab import files
    files.download(filename)
except ImportError:
    pass

/tmp/ipykernel_322/2079689449.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  mmm = yf.download("MMM", start="2000-01-01", end="2010-12-31")
[*********************100%***********************]  1 of 1 completed


Success! Results and interpretations saved to MMM_ARCH_Model_Selection.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>